# fact_player_snapshot.py

## Purpose
Builds `fact_player_snapshot` as the fact table with per-snapshot metrics for each player.

## Sources
- Bronze: `fifa.bronze.ea_fc_players_raw`
- Silver: `dim_edition`

## Grain
- One row per `(player_id, snapshot_date)`

## Unique key
- `player_snapshot_key` = `player_id + snapshot_date`

## Strategy
- First load: `overwrite`
- Subsequent loads: `MERGE` on `player_snapshot_key`
- Partitioned by `snapshot_date` for efficient date-filtered queries

## Metrics
- Card facets: `pace`, `shooting`, `passing`, `dribbling`, `defending`, `physicality`
- `overall_rating`
- 34 detailed attributes (`attacking_*`, `skill_*`, `movement_*`, `power_*`, `mentality_*`, `defending_*`, `goalkeeping_*`)

## Referenced dimensions
- `player_id` → `dim_player`
- `edition_id` → `dim_edition`
- `snapshot_date` → derived from Bronze (part of the grain)

## Output
- Table: `fifa.silver.fact_player_snapshot`

## Columns

### Keys
- `player_snapshot_key` (STRING)
- `player_id` (INT)
- `edition_id` (INT)
- `snapshot_date` (DATE)

### Card facets
- `overall_rating` (INT)
- `pace` (INT)
- `shooting` (INT)
- `passing` (INT)
- `dribbling` (INT)
- `defending` (INT)
- `physicality` (INT)

### Detailed attributes (grouped by facet)

See the [Metric Grouping](#metric-grouping) section for the full breakdown.

### Metadata
- `_ingestion_timestamp` (TIMESTAMP)

## Metric Grouping

The 34 detailed attributes are grouped into 6 card facets, following EA's own grouping. This reflects how the attributes are displayed on the player card in-game.

| Facet | Source Group | Attributes |
|---|---|---|
| **Pace** | Movement | `movement_acceleration`, `movement_sprint_speed` |
| **Shooting** | Attacking | `attacking_finishing`, `attacking_heading_accuracy`, `attacking_volleys` |
| | Power | `power_shot_power`, `power_long_shots` |
| | Mentality | `mentality_penalties` |
| **Passing** | Attacking | `attacking_crossing`, `attacking_short_passing` |
| | Skill | `skill_curve`, `skill_fk_accuracy`, `skill_long_passing` |
| | Mentality | `mentality_vision` |
| **Dribbling** | Skill | `skill_dribbling`, `skill_ball_control` |
| | Movement | `movement_agility`, `movement_reactions`, `movement_balance` |
| **Defending** | Defending | `defending_awareness`, `defending_standing_tackle`, `defending_sliding_tackle` |
| | Mentality | `mentality_interceptions`, `mentality_positioning`, `mentality_aggression`, `mentality_composure` |
| **Physicality** | Power | `power_jumping`, `power_stamina`, `power_strength` |

### Goalkeeping attributes

Goalkeepers have their own set of attributes, which are not part of the 6 card facets:

- `goalkeeping_diving`
- `goalkeeping_handling`
- `goalkeeping_kicking`
- `goalkeeping_positioning`
- `goalkeeping_reflexes`

### Why this grouping matters

When using the `fact_player_facets` table (for the radar chart in Power BI), the attributes are unpivoted into rows. Grouping them by facet in the documentation makes it easier to understand which attributes belong to which facet.

**Note**: EA has changed the grouping of some attributes between editions. This grouping reflects FC 27. If future editions change it, this table should be updated.

## Why `player_snapshot_key`?

`player_id` alone is not unique in this table (a player has one row per snapshot). The composite key `player_id + snapshot_date` solves two problems:

1. **Power BI relationships**: it gives a unique key for the relationship between `dim_player` and `fact_player_snapshot`.
2. **Idempotency**: the MERGE can safely update rows without duplicating them.

## Why metrics don't go in `dim_player`

The 34 detailed attributes are **measures**, not descriptors. They change with every snapshot (patch updates). Putting them in the dimension would duplicate them across SCD2 versions.

This is the standard **star schema** pattern: descriptive attributes in the dimension, measures in the fact table.

## Why `overwrite` on the first load and `MERGE` afterwards?

- **First load**: the table doesn't exist, so `overwrite` creates it with the correct schema.
- **Subsequent loads**: `MERGE` updates existing rows and inserts new ones without rewriting the entire table.

## Partitioning

The table is partitioned by `snapshot_date`. This speeds up date-filtered queries and lets the MERGE touch only the affected partitions.

## Incremental load: how new snapshots are handled

When a new snapshot arrives (e.g. a patch update), the pipeline:

1. Reads Bronze (which now contains the new snapshot).
2. Builds `fact_enriched` with the new `player_snapshot_key` values.
3. Runs the MERGE:
   - If the `player_snapshot_key` already exists → **update** the metrics.
   - If it doesn't exist → **insert** the new row.

No rows are ever duplicated.

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, concat
from delta.tables import DeltaTable

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

fact = (
    bronze
    .select(
        col("player_id").cast("int"),
        col("edition").alias("edition_code"),
        to_date(col("snapshot_date"), "yyyy-MM-dd").alias("snapshot_date"),
        col("overall_rating").cast("int"),
        col("pace").cast("int"),
        col("shooting").cast("int"),
        col("passing").cast("int"),
        col("dribbling").cast("int"),
        col("defending").cast("int"),
        col("physicality").cast("int"),
        col("attacking_crossing").cast("int"),
        col("attacking_finishing").cast("int"),
        col("attacking_heading_accuracy").cast("int"),
        col("attacking_short_passing").cast("int"),
        col("attacking_volleys").cast("int"),
        col("skill_dribbling").cast("int"),
        col("skill_curve").cast("int"),
        col("skill_fk_accuracy").cast("int"),
        col("skill_long_passing").cast("int"),
        col("skill_ball_control").cast("int"),
        col("movement_acceleration").cast("int"),
        col("movement_sprint_speed").cast("int"),
        col("movement_agility").cast("int"),
        col("movement_reactions").cast("int"),
        col("movement_balance").cast("int"),
        col("power_shot_power").cast("int"),
        col("power_jumping").cast("int"),
        col("power_stamina").cast("int"),
        col("power_strength").cast("int"),
        col("power_long_shots").cast("int"),
        col("mentality_aggression").cast("int"),
        col("mentality_interceptions").cast("int"),
        col("mentality_positioning").cast("int"),
        col("mentality_vision").cast("int"),
        col("mentality_penalties").cast("int"),
        col("mentality_composure").cast("int"),
        col("defending_awareness").cast("int"),
        col("defending_standing_tackle").cast("int"),
        col("defending_sliding_tackle").cast("int"),
        col("goalkeeping_diving").cast("int"),
        col("goalkeeping_handling").cast("int"),
        col("goalkeeping_kicking").cast("int"),
        col("goalkeeping_positioning").cast("int"),
        col("goalkeeping_reflexes").cast("int")
    )
    .filter(col("edition_code").isNotNull() & (col("edition_code") != ""))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .distinct()
)

dim_edition = spark.read.table("fifa.silver.dim_edition")

fact_enriched = (
    fact
    .join(
        dim_edition.select("edition_id", "edition_code"),
        on="edition_code",
        how="left"
    )
    # Composite key: player_id + snapshot_date
    .withColumn("player_snapshot_key", concat(col("player_id"), lit("-"), col("snapshot_date")))
    .select(
        "player_snapshot_key",
        "player_id", "edition_id", "snapshot_date",
        "overall_rating", "pace", "shooting", "passing", "dribbling", "defending", "physicality",
        "attacking_crossing", "attacking_finishing", "attacking_heading_accuracy",
        "attacking_short_passing", "attacking_volleys",
        "skill_dribbling", "skill_curve", "skill_fk_accuracy", "skill_long_passing", "skill_ball_control",
        "movement_acceleration", "movement_sprint_speed", "movement_agility",
        "movement_reactions", "movement_balance",
        "power_shot_power", "power_jumping", "power_stamina", "power_strength", "power_long_shots",
        "mentality_aggression", "mentality_interceptions", "mentality_positioning",
        "mentality_vision", "mentality_penalties", "mentality_composure",
        "defending_awareness", "defending_standing_tackle", "defending_sliding_tackle",
        "goalkeeping_diving", "goalkeeping_handling", "goalkeeping_kicking",
        "goalkeeping_positioning", "goalkeeping_reflexes",
        "_ingestion_timestamp"
    )
)

# ===== Create Silver table =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.fact_player_snapshot (
    player_snapshot_key STRING,
    player_id INT,
    edition_id INT,
    snapshot_date DATE,
    overall_rating INT,
    pace INT,
    shooting INT,
    passing INT,
    dribbling INT,
    defending INT,
    physicality INT,
    attacking_crossing INT,
    attacking_finishing INT,
    attacking_heading_accuracy INT,
    attacking_short_passing INT,
    attacking_volleys INT,
    skill_dribbling INT,
    skill_curve INT,
    skill_fk_accuracy INT,
    skill_long_passing INT,
    skill_ball_control INT,
    movement_acceleration INT,
    movement_sprint_speed INT,
    movement_agility INT,
    movement_reactions INT,
    movement_balance INT,
    power_shot_power INT,
    power_jumping INT,
    power_stamina INT,
    power_strength INT,
    power_long_shots INT,
    mentality_aggression INT,
    mentality_interceptions INT,
    mentality_positioning INT,
    mentality_vision INT,
    mentality_penalties INT,
    mentality_composure INT,
    defending_awareness INT,
    defending_standing_tackle INT,
    defending_sliding_tackle INT,
    goalkeeping_diving INT,
    goalkeeping_handling INT,
    goalkeeping_kicking INT,
    goalkeeping_positioning INT,
    goalkeeping_reflexes INT,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA
PARTITIONED BY (snapshot_date);
""")

# ===== Fact table load: overwrite on first load, MERGE afterwards =====
target_table = "fifa.silver.fact_player_snapshot"

table_exists = spark.catalog.tableExists(target_table)
row_count = 0

if table_exists:
    row_count = spark.read.table(target_table).count()

if not table_exists or row_count == 0:
    # First load: overwrite
    print("First load detected. Writing with overwrite...")
    (fact_enriched
     .write
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(target_table)
    )
    print(f"Table created with {spark.read.table(target_table).count()} rows")

else:
    # Subsequent loads: MERGE by player_snapshot_key
    print(f"Existing table with {row_count} rows. Applying MERGE...")

    target = DeltaTable.forName(spark, target_table)

    # Metric columns to update on match
    metric_columns = [
        "overall_rating", "pace", "shooting", "passing", "dribbling", "defending", "physicality",
        "attacking_crossing", "attacking_finishing", "attacking_heading_accuracy",
        "attacking_short_passing", "attacking_volleys",
        "skill_dribbling", "skill_curve", "skill_fk_accuracy", "skill_long_passing", "skill_ball_control",
        "movement_acceleration", "movement_sprint_speed", "movement_agility",
        "movement_reactions", "movement_balance",
        "power_shot_power", "power_jumping", "power_stamina", "power_strength", "power_long_shots",
        "mentality_aggression", "mentality_interceptions", "mentality_positioning",
        "mentality_vision", "mentality_penalties", "mentality_composure",
        "defending_awareness", "defending_standing_tackle", "defending_sliding_tackle",
        "goalkeeping_diving", "goalkeeping_handling", "goalkeeping_kicking",
        "goalkeeping_positioning", "goalkeeping_reflexes"
    ]

    update_set = {c: f"s.{c}" for c in metric_columns}
    update_set["edition_id"] = "s.edition_id"
    update_set["_ingestion_timestamp"] = "s._ingestion_timestamp"

    insert_values = {
        "player_snapshot_key": "s.player_snapshot_key",
        "player_id": "s.player_id",
        "edition_id": "s.edition_id",
        "snapshot_date": "s.snapshot_date",
        "_ingestion_timestamp": "s._ingestion_timestamp"
    }
    insert_values.update({c: f"s.{c}" for c in metric_columns})

    (
        target.alias("t")
        .merge(
            fact_enriched.alias("s"),
            "t.player_snapshot_key = s.player_snapshot_key"
        )
        .whenMatchedUpdate(set=update_set)
        .whenNotMatchedInsert(values=insert_values)
        .execute()
    )

    print(f"MERGE completed. Total rows: {spark.read.table(target_table).count()}")

LIMIT 10 of the fact_player_snapshot table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
select 
  *
from fifa.silver.fact_player_snapshot order by player_snapshot_key limit 10;